# 02 — Preprocessing
**Home Credit Default Risk — Capstone Step 2 (part 1)**

Objective: handle sentinel values, document missing-data decisions, create missingness indicators, and impute.

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = '../data/'
df = pd.read_csv(DATA_DIR + 'application_train.csv')
print(df.shape)


(307511, 122)


## 1. Handle Sentinel Values

`DAYS_EMPLOYED = 365243` is a sentinel meaning unemployed/retired, not 1000 years of employment. `OWN_CAR_AGE` missingness is structural (no car), confirmed in Step 1: 99.998% of missing values correspond to `FLAG_OWN_CAR == 'N'`.

In [2]:
# DAYS_EMPLOYED sentinel -> NaN
n_sentinel = (df['DAYS_EMPLOYED'] == 365243).sum()
print(f"DAYS_EMPLOYED sentinel (365243) count: {n_sentinel}")

DAYS_EMPLOYED sentinel (365243) count: 55374


In [3]:
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

In [4]:
df.loc[df['FLAG_OWN_CAR'] == 'N', 'OWN_CAR_AGE'] = -1

# Remaining missing OWN_CAR_AGE (car owners with no recorded age) -> median among car owners
median_car_age = df.loc[df['FLAG_OWN_CAR'] == 'Y', 'OWN_CAR_AGE'].median()
df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].fillna(median_car_age)

print("Remaining OWN_CAR_AGE nulls:", df['OWN_CAR_AGE'].isnull().sum())

Remaining OWN_CAR_AGE nulls: 0


**Rationale:** 
- `DAYS_EMPLOYED`'s sentinel is converted to NaN so it doesn't distort imputation/modeling downstream (it will be imputed like any other missing value in Section 3, informed by a missingness indicator since it's >5% missing). 

- `OWN_CAR_AGE` uses -1 for "no car" (0 is a plausible real value for a brand-new car, so it can't be used as the sentinel) rather than the population median, to avoid falsely implying non-owners have a car of a typical age.

### Check for other sentinel values

Look for suspiciously round or extreme values in other DAYS_* and numeric columns.

In [7]:
days_cols = [c for c in df.columns if c.startswith('DAYS_')]
for c in days_cols:
    print(c, '-> min:', df[c].min(), 'max:', df[c].max())

DAYS_BIRTH -> min: -25229 max: -7489
DAYS_EMPLOYED -> min: -17912.0 max: 0.0
DAYS_REGISTRATION -> min: -24672.0 max: 0.0
DAYS_ID_PUBLISH -> min: -7197 max: 0
DAYS_LAST_PHONE_CHANGE -> min: -4292.0 max: 0.0


**Interpretation:**

No other suspicious sentinel values were found among the DAYS_* columns — all ranges are consistent with dates counted backward from the application date (negative or zero). DAYS_EMPLOYED's max of 0.0 (rather than 365243) confirms the sentinel cleanup in Section 1 was applied successfully.

## 2. Missing Data Decisions (>40% missing)

For each feature with more than 40% missing, decide: drop, impute, or keep with indicator.

In [10]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
high_missing = missing_pct[missing_pct > 40]
print(f"Columns with >40% missing: {len(high_missing)}")
high_missing

Columns with >40% missing: 48


COMMONAREA_AVG                  69.872297
COMMONAREA_MODE                 69.872297
COMMONAREA_MEDI                 69.872297
NONLIVINGAPARTMENTS_MEDI        69.432963
NONLIVINGAPARTMENTS_MODE        69.432963
NONLIVINGAPARTMENTS_AVG         69.432963
FONDKAPREMONT_MODE              68.386172
LIVINGAPARTMENTS_AVG            68.354953
LIVINGAPARTMENTS_MEDI           68.354953
LIVINGAPARTMENTS_MODE           68.354953
FLOORSMIN_MODE                  67.848630
FLOORSMIN_AVG                   67.848630
FLOORSMIN_MEDI                  67.848630
YEARS_BUILD_AVG                 66.497784
YEARS_BUILD_MODE                66.497784
YEARS_BUILD_MEDI                66.497784
LANDAREA_MEDI                   59.376738
LANDAREA_MODE                   59.376738
LANDAREA_AVG                    59.376738
BASEMENTAREA_AVG                58.515956
BASEMENTAREA_MODE               58.515956
BASEMENTAREA_MEDI               58.515956
EXT_SOURCE_1                    56.381073
NONLIVINGAREA_MODE              55

**Rationale (documented decisions):**

- **`EXT_SOURCE_1`** (56.4% missing): despite the high missingness, this is one of the three most predictive features in the dataset (confirmed in Step 1's correlation analysis, r=0.155 with TARGET).  
Decision: **keep, with a missingness indicator + median imputation** — dropping it would discard the single strongest linear signal in the data. This is treated as a special case, distinct from the property columns below.

- **Property/building columns** (`COMMONAREA_*`, `NONLIVINGAPARTMENTS_*`, `LIVINGAPARTMENTS_*`, `FLOORSMIN_*`, `YEARS_BUILD_*`, `LANDAREA_*`, `BASEMENTAREA_*`, `NONLIVINGAREA_*`, `YEARS_BEGINEXPLUATATION_*`, `TOTALAREA_MODE`, `FONDKAPREMONT_MODE`, all ~48-70% missing): these describe apartment/building characteristics that likely don't apply to renters or aren't consistently collected.  
Decision: **keep with a missingness indicator** (tested in Section 3) rather than drop, since absence may itself correlate with housing type — a weaker but still plausible signal. Numeric values are median-imputed after the indicator is created.

- **`EMERGENCYSTATE_MODE`** (47.4% missing): the only categorical column in this high-missing group.  
Decision: **keep with a missingness indicator**, mode-imputed (or encoded with an explicit "Unknown" category) rather than dropped, following the same reasoning as the other property-related fields.

## 3. Missingness Indicators (>5% missing)

Create a binary indicator for every column with more than 5% missing, then test whether missingness itself is predictive of TARGET.

In [11]:
cols_gt5_missing = missing_pct[missing_pct > 5].index.tolist()
print(f"Columns with >5% missing (will get an indicator): {len(cols_gt5_missing)}")

Columns with >5% missing (will get an indicator): 57


In [12]:
for col in cols_gt5_missing:
    df[f'{col}_WAS_MISSING'] = df[col].isnull().astype(int)

indicator_cols = [f'{col}_WAS_MISSING' for col in cols_gt5_missing]
print(f"Created {len(indicator_cols)} missingness indicator columns")

Created 57 missingness indicator columns


Est-ce que savoir QUELLES colonnes sont manquantes pour une personne aide à prédire si elle va faire défaut ?

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X_ind = df[indicator_cols].fillna(0)
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X_ind, y, test_size=0.2, stratify=y, random_state=42
)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train, y_train)
preds = clf.predict_proba(X_test)[:, 1]
auroc = roc_auc_score(y_test, preds)
print(f"AUROC using missingness indicators only: {auroc:.4f}")

AUROC using missingness indicators only: 0.5783


**Interpretation:**

Using missingness indicators alone (no actual feature values), a logistic regression achieves an AUROC of 0.5783 — meaningfully above the 0.5 random baseline. This confirms that the *pattern* of which fields are missing (property characteristics, EXT_SOURCE_1, OWN_CAR_AGE, etc.) carries genuine predictive signal about default risk, independent of the underlying values. This justifies keeping the `_WAS_MISSING` indicator columns as model features going forward, rather than discarding them after imputation.

## 4. Imputation

Impute remaining numeric features with median, categorical features with mode.

In [14]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

Numeric columns: 163
Categorical columns: 16


In [15]:
# Don't impute TARGET or ID columns
numeric_cols = [c for c in numeric_cols if c not in ['TARGET', 'SK_ID_CURR']]

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining nulls after imputation:", df.isnull().sum().sum())

Remaining nulls after imputation: 0


**Rationale:**  
* Median imputation is used for numeric features because most (e.g. `AMT_INCOME_TOTAL`, `AMT_CREDIT`) are right-skewed — median is robust to outliers, unlike the mean. Mode imputation is used for categoricals since there's no meaningful "average" category. The missingness indicators created in Section 3 preserve the *information* that a value was originally missing, even after this generic imputation fills in a placeholder value.

## 5. Save Preprocessed Data

In [16]:
import os
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/preprocessed_train.csv', index=False)
print("Saved:", df.shape)

Saved: (307511, 179)


## Note: AMT_ANNUITY Unit Clarification

EDA (Step 1) flagged a possible unit mismatch: treating `AMT_ANNUITY` as monthly (per the finance background guide's suggested formula `AMT_ANNUITY / (AMT_INCOME_TOTAL / 12)`) implies a median DTI over 200%, which is unrealistic.

Two pieces of evidence from community solutions to this same Kaggle competition support treating `AMT_ANNUITY` as an **annual** figure (same scale as `AMT_INCOME_TOTAL`, not divided by 12):

1. A widely-used feature formula computes `ANNUITY_INCOME_RATIO = AMT_ANNUITY / AMT_INCOME_TOTAL` directly, with no `/12` adjustment on either side.
2. A top-placing Kaggle solution defines `loan payment length = AMT_CREDIT / AMT_ANNUITY`. For this dataset's median values (~513,531 / ~24,903 ≈ 20.6), that only makes sense as a loan term in **years** (~20.6 years, plausible for a large loan) — not months (~20.6 months, implausibly short for a 500K+ loan).

**Decision:** `ANNUITY_INCOME_RATIO` will be computed in `03_feature_engineering.ipynb` as `AMT_ANNUITY / AMT_INCOME_TOTAL` (no `/12`).